# SAC Irrigation Training - v2.16 (rain-blindness fix: RAIN_REF=30 + capped auto-α, Kaggle)

**Algorithm:** SAC (stable_baselines3) with VDN-factorised twin-Q + LayerNorm critic (architecturally byte-identical to v2.15)
**Actor:** LeakyReLU(0.01) MLP, trained on NORMALISED global/forecast observations (with tighter rainfall scaling)
**Learning rate:** asymmetric - actor LR = 5x critic LR
**Entropy:** auto-tuned, **capped at 0.1**, initialised at 0.05, target_entropy = -65 (vs SB3 default -130)
**Reward r6:** LINEAR (carried from v2.15)
**rain_normaliser:** **30.0 mm/day** (v2.15 used 70.0)

## Why v2.16 exists

v2.15 (linear r6) trained stably but did not improve over v2.14. Direct
gradient analysis of v2.15's 250k actor on realistic observations revealed
the root cause of wet-year over-irrigation:

| feature                | gradient | input range | effective sensitivity |
|------------------------|----------|-------------|----------------------|
| rain forecast          | 0.43     | 0.36        | **0.16** (lowest)    |
| ETc forecast           | 1.24     | 1.44        | 1.78                 |
| rad forecast           | 1.40     | 1.56        | 2.19                 |
| soil moisture (x1)     | 2.03     | 3.00        | 6.08                 |

The actor IS responsive to ET and radiation forecasts. Rain has comparable
gradient but ~10× less effective sensitivity because the *input never moves*:
rainfall is divided by RAIN_REF=70 mm/day, while empirical median rain is
0.07 mm (median normalized = 0.001, p99 = 0.18). After 2x-1 recentering, rain
inputs occupy only the bottom 0.36 of [-1, +1] vs ET/rad spanning 1.4-1.6.

This is the **rain-blindness** diagnostic — it explains corr(u, rain_fwd7)
≈ +0.03 in v2.14/v2.15 (vs MPC's -0.42).

## Two paired changes in v2.16

**1. RAIN_REF: 70.0 → 30.0**
Evaluated against the 26-year growing-season climate record:

| ref | train p99/ref | 2024 max/ref | train %clip | 2024 %clip | recentered span |
|-----|---------------|--------------|-------------|------------|-----------------|
| 15  | 0.83          | 2.39         | 0.75%       | 2.15%      | 1.66            |
| **30**  | **0.42**          | **1.19**         | **0.09%**       | **1.08%**      | **0.83**            |
| 70  | 0.18          | 0.51         | 0.00%       | 0.00%      | 0.36 (current)  |

RAIN_REF=30 was chosen over alternatives because RAIN_REF=15 clips 2.15% of
2024 wet-year days (including 14, 19, 35 mm events that distinguish wet from
moderate years), while RAIN_REF=30 preserves the full operational range up
to 30 mm and only clips the single 36 mm outlier (0.16% of all days).

**2. ent_coef: 0.01 fixed → auto-tuned, capped at 0.1, target_entropy=-65**
SB3's default target_entropy is `-dim(action) = -130`, derived for monolithic
action spaces. Our actor is VDN-factorised: 130 actions are produced by a
shared per-agent network, so per-cell entropy is the actual control variable.
Target -65 = -0.5 per cell, encouraging moderate per-cell stochasticity
(better for shared-actor buffer diversity than -130 = -1.0 per cell).
Initial log_ent_coef at log(0.05) ≈ -3.0 (v2.14/v2.15 known-stable). Cap at
log(0.1) ≈ -2.303 enforced by post-step parameter clipping.

Architecturally byte-identical to v2.15. All other hyperparameters unchanged.


In [ ]:
# Cell 1: Clone repo and install deps.
import subprocess, sys, os

WORK = '/kaggle/working'
repo = os.path.join(WORK, 'thesis')
if os.path.exists(repo):
    subprocess.run(['rm', '-rf', repo], check=True)
subprocess.run(
    ['git', 'clone', 'https://github.com/taratorbati/thesis.git', repo],
    check=True)

os.chdir(repo)
sys.path.insert(0, repo)

subprocess.run(
    ['pip', 'install', '--quiet',
     'stable-baselines3==2.6.0', 'gymnasium', 'wandb', 'pytest'],
    check=True)

import torch
print(f'PyTorch:        {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:            {torch.cuda.get_device_name(0)}')


In [ ]:
# Cell 2: WandB secret + GPU check.
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
    print('OK  WANDB_API_KEY loaded from Kaggle Secrets.')
except Exception as e:
    print(f'NOTE: Could not load WANDB_API_KEY ({type(e).__name__}).')
    print('     Training continues without WandB - add it via Add-ons > Secrets to enable it.')

import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'nvidia-smi failed - no GPU allocated')


In [ ]:
# Cell 3: Pre-training validation.
import subprocess, sys

print('Smoke tests...')
r = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_rl_smoke.py', '-v', '--tb=short'],
    capture_output=False)
assert r.returncode == 0, 'SMOKE TESTS FAILED'

print('\nFactorized-critic tests (v2.7 through v2.16)...')
r = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_factorized_critic.py', '-v', '--tb=short'],
    capture_output=False)
assert r.returncode == 0, 'FACTORIZED CRITIC TESTS FAILED'

print('\n1000-step pilot training (wiring check, ~1-2 min)...')
from src.rl.train_v216 import train_sac_v216
_ = train_sac_v216(
    seed=999,
    output_dir='/kaggle/working/pilot',
    wandb_project=None,
    total_timesteps=1000,
)
print('\nOK  Pre-flight passed. Proceed to Cell 4.')


In [ ]:
# Cell 4: Full 250k training (SAC v2.16 - RAIN_REF=30 + capped auto-α).
# ~30-55 min on A100, ~2-2.5 h on T4.
#
# Start with SEED=0 (paired with v2.7 / v2.11 / v2.14 / v2.15 seed 0). Expand
# to seeds 1, 2 only after seed-0 meets the primary acceptance criteria.

SEED = 0       # CHANGE per session

from src.rl.train_v216 import train_sac_v216

model = train_sac_v216(
    seed=SEED,
    output_dir='/kaggle/working/thesis/results/rl',
    wandb_project='sac-irrigation-thesis',
    total_timesteps=250_000,
    gamma=0.99,
    actor_lr_mult=5.0,
    ent_coef_init='auto_0.05',          # auto-tune, init at v2.14/v2.15 level
    ent_coef_cap=0.1,                   # *** THE v2.16 alpha cap ***
    target_entropy=-65.0,               # *** -65 vs SB3 default -130 ***
    reward_overshoot_mode='linear',     # carried from v2.15
    rain_normaliser=30.0,               # *** THE v2.16 rain change ***
)
print('Training complete.')


In [ ]:
# Cell 5: Archive results so Kaggle persists them after the session.
import shutil, os, datetime

src = f'/kaggle/working/thesis/results/rl/sac_v216_seed{SEED}'
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
dst = f'/kaggle/working/sac_v216_seed{SEED}_{timestamp}'

shutil.copytree(src, dst, ignore=shutil.ignore_patterns('replay_buffer_latest.pkl'))
print(f'Archived to: {dst}  (download from the Kaggle output panel)')
for root, _, files in os.walk(dst):
    for f in files:
        p = os.path.join(root, f); size = os.path.getsize(p)
        print(f'  {os.path.relpath(p, dst)}  ({size/1024:.1f} KB)')


In [ ]:
# Cell 6: Post-training 9-cell evaluation (SAC eval path).
#
# v2.16 produces a SAC checkpoint with marker=2.16. The runner auto-detects it
# via the 'actor.obs_norm_marker' buffer and dispatches to V216CTDESACPolicy,
# AND applies rain_normaliser=30.0 at eval time (matching training).
import subprocess, sys

model_path = f'/kaggle/working/thesis/results/rl/sac_v216_seed{SEED}/best_model/best_model.zip'

print('Evaluating on 9-cell grid (perfect forecast)...')
r = subprocess.run([
    sys.executable, '-m', 'scripts.experiments.exp_rl',
    '--mode',     'eval',
    '--model',    model_path,
    '--scenario', 'all',
    '--budget',   'all',
    '--forecast', 'perfect',
], capture_output=False)
assert r.returncode == 0, 'PERFECT-FORECAST EVAL FAILED'

# Also evaluate the FINAL checkpoint - v2.15 showed the final policy was more
# responsive than EvalCallback's "best".  Compare both.
final_path = f'/kaggle/working/thesis/results/rl/sac_v216_seed{SEED}/sac_v216_seed{SEED}_final.zip'
import os
if os.path.exists(final_path):
    print('\nEvaluating FINAL checkpoint (250k) for comparison with best_model...')
    r = subprocess.run([
        sys.executable, '-m', 'scripts.experiments.exp_rl',
        '--mode',     'eval',
        '--model',    final_path,
        '--scenario', 'all',
        '--budget',   'all',
        '--forecast', 'perfect',
        '--output-dir', f'/kaggle/working/thesis/results/runs/sac_v216_final_model',
    ], capture_output=False)


In [ ]:
# Cell 7: PRIMARY DIAGNOSTIC - "did rain-blindness lift?"
#
# This is the v2.16 acceptance check. v2.15's primary defect was that
# corr(u, rain_fwd7) ≈ +0.03 (essentially no rain forecast response) and
# wet/100 u_p10 = 2.5 mm. If RAIN_REF=30 worked, the forecast correlation
# should become substantially more negative.
import pandas as pd, numpy as np, glob, os

OUTPUT_DIR = '/kaggle/working/thesis/results/runs/sac_v216_best_model'
if not os.path.isdir(OUTPUT_DIR):
    # The runner may save under a different layout - search:
    cands = glob.glob('/kaggle/working/thesis/results/runs/sac_v216*')
    print('Result dirs found:', cands)
    OUTPUT_DIR = cands[0] if cands else 'results/runs/sac_v216_best_model'

# Locate the wet/100 parquet
candidates = glob.glob(os.path.join(OUTPUT_DIR, 'sac_perfect_det_wet_rice_100pct_seed0.parquet'))
if not candidates:
    candidates = glob.glob(os.path.join(OUTPUT_DIR, '*wet*100*seed0.parquet'))
assert candidates, f'No wet/100 parquet found in {OUTPUT_DIR}'

df = pd.read_parquet(candidates[0])
g = df.groupby('day').agg(u=('u','mean'), x1=('x1','mean'),
                          rain=('rainfall','mean'), et=('et0','mean'),
                          x5=('x5','mean'), bud=('budget_remaining','mean'))
g['rain_fwd7'] = g['rain'].rolling(7).sum().shift(-7)
g['day_idx']   = g.index

# Compute the v2.15-comparison diagnostics
print('=' * 78)
print(' v2.16 wet/100 BEHAVIOURAL DIAGNOSTIC vs v2.15 vs MPC')
print('=' * 78)
print(f'{"metric":<32s} {"v2.16":>10s} {"v2.15":>10s} {"v2.14":>10s} {"MPC":>10s}')
print('-' * 78)
def fmt(v, w=10, p=4):
    return f'{v:>{w}.{p}f}' if v is not None and not (isinstance(v, float) and np.isnan(v)) else f'{"NA":>{w}}'
metrics = {
    'u mean (mm)':            (g['u'].mean(),                      4.853,  4.323,  3.336),
    'u daily 10th-pctile':    (g['u'].quantile(0.10),              2.501,  3.310,  0.161),
    'u daily min':            (g['u'].min(),                       0.281,  2.646,  0.026),
    'u daily max':            (g['u'].max(),                       9.415,  5.495,  7.936),
    'corr(u, rain_today)':    (g['u'].corr(g['rain']),            -0.275, -0.163, -0.390),
    'corr(u, rain_fwd7)':     (g['u'].corr(g['rain_fwd7']),        0.034, -0.109, -0.419),
    'corr(u, x1)':            (g['u'].corr(g['x1']),              -0.172, -0.617, -0.242),
    'corr(u, et)':            (g['u'].corr(g['et']),               0.144,  0.363,  0.488),
}
for k, vals in metrics.items():
    v16, v15, v14, mpc = vals
    print(f'{k:<32s} {fmt(v16)} {fmt(v15)} {fmt(v14)} {fmt(mpc)}')

print()
print('Primary acceptance:')
rfc = g['u'].corr(g['rain_fwd7'])
print(f"  corr(u, rain_fwd7) = {rfc:+.4f}")
print(f"  PASS if rfc < -0.10  (v2.15: +0.03; MPC: -0.42)")
print(f"  -> {'PASS' if rfc < -0.10 else 'FAIL'}")

rt = g['u'].corr(g['rain'])
print(f"  corr(u, rain_today) = {rt:+.4f}")
print(f"  PASS if rt < -0.30  (v2.15: -0.28)")
print(f"  -> {'PASS' if rt < -0.30 else 'FAIL'}")


In [ ]:
# Cell 8: v2.16 vs v2.15 vs v2.14 vs MPC - 9-cell summary.
import json, glob, os, numpy as np

ORDER = [('dry','100pct'),('dry','85pct'),('dry','70pct'),
         ('moderate','100pct'),('moderate','85pct'),('moderate','70pct'),
         ('wet','100pct'),('wet','85pct'),('wet','70pct')]

def grab(pattern):
    res = {}
    for f in sorted(glob.glob(pattern)):
        if 'seed1' in os.path.basename(f): continue
        j = json.load(open(f)); fn = os.path.basename(f)
        pct = [p for p in ['100pct','85pct','70pct'] if p in fn][0]
        m = j['final_metrics']
        res[(j['scenario'],pct)] = (m['yield_kg_ha'], m.get('water_used_mm'),
                                     m.get('waterlog_days_per_agent'),
                                     m.get('wue_kg_ha_per_mm'))
    return res

# v2.16 (just trained):
v216 = grab(os.path.join('/kaggle/working/thesis/results/runs/sac_v216_best_model',
                          'sac_perfect_det_*seed0.json'))
if not v216:
    v216 = grab('results/runs/sac_v216_best_model/sac_perfect_det_*seed0.json')

# v2.15 / v2.14 / MPC (baselines from repo):
v215 = grab('results/runs/sac_v215_best_model/sac_perfect_det_*seed0.json')
v214 = grab('results/runs/sac_v214_best_model/sac_perfect_det_*seed0.json')
mpc  = grab('results/runs/mpc_perfect_*_Hp8.json')

def mean(d, i):
    vs = [d[o][i] for o in ORDER if o in d and d[o][i] is not None]
    return float(np.mean(vs)) if vs else float('nan')

print('='*100)
print(' v2.16 vs v2.15 vs v2.14 vs MPC  -  9-cell perfect-forecast grid, seed 0')
print('='*100)
print(f'%-20s %15s %15s %15s %15s' % ('scenario','v2.16 y/mm/wlog','v2.15 y/mm/wlog','v2.14 y/mm/wlog','MPC y/mm/wlog'))
for o in ORDER:
    def fmt(d):
        if o not in d: return 'NA'
        y,w,wl,_ = d[o]; return f'{y:.0f}/{w or 0:.0f}/{wl or 0:.0f}'
    print(f'%-20s %15s %15s %15s %15s' % (o[0]+'/'+o[1], fmt(v216), fmt(v215), fmt(v214), fmt(mpc)))
print('-'*100)
print(f'MEAN yield      v2.16={mean(v216,0):7.1f}  v2.15={mean(v215,0):7.1f}  v2.14={mean(v214,0):7.1f}  MPC={mean(mpc,0):7.1f}')
print(f'MEAN water      v2.16={mean(v216,1):7.1f}  v2.15={mean(v215,1):7.1f}  v2.14={mean(v214,1):7.1f}  MPC={mean(mpc,1):7.1f}')
print(f'MEAN waterlog   v2.16={mean(v216,2):7.2f}  v2.15={mean(v215,2):7.2f}  v2.14={mean(v214,2):7.2f}  MPC={mean(mpc,2):7.2f}')
print(f'MEAN WUE        v2.16={mean(v216,3):7.2f}  v2.15={mean(v215,3):7.2f}  v2.14={mean(v214,3):7.2f}  MPC={mean(mpc,3):7.2f}')

# Wet-year focus (the regime that matters):
print()
def wet_mean(d, i):
    vs = [d[o][i] for o in ORDER if o[0]=='wet' and o in d and d[o][i] is not None]
    return float(np.mean(vs)) if vs else float('nan')
print(f'WET-only yield  v2.16={wet_mean(v216,0):7.1f}  v2.15={wet_mean(v215,0):7.1f}  v2.14={wet_mean(v214,0):7.1f}  MPC={wet_mean(mpc,0):7.1f}')
print(f'WET-only water  v2.16={wet_mean(v216,1):7.1f}  v2.15={wet_mean(v215,1):7.1f}  v2.14={wet_mean(v214,1):7.1f}  MPC={wet_mean(mpc,1):7.1f}')
print(f'WET-only wlog   v2.16={wet_mean(v216,2):7.2f}  v2.15={wet_mean(v215,2):7.2f}  v2.14={wet_mean(v214,2):7.2f}  MPC={wet_mean(mpc,2):7.2f}')

print()
print('Acceptance:')
print(f'  - Wet-year water drops:        v2.16 wet_water_mean = {wet_mean(v216,1):.0f} mm '
      f'(v2.15 = 408; v2.14 = 408; target < 380; MPC = 308)')
print(f'  - Wet-year yield rises:        v2.16 wet_yield_mean = {wet_mean(v216,0):.0f} kg/ha '
      f'(v2.15 = 3392; v2.14 = 3444; target >= 3450)')
print(f'  - Dry-year yield stays high:   v2.16 dry_yield_mean = '
      f'{np.mean([v216[o][0] for o in ORDER if o[0]=="dry" and o in v216]):.0f} kg/ha '
      f'(v2.14 = 3975; floor 3900)')


In [ ]:
# Cell 9: Alpha trajectory diagnostic.
#
# Plots the auto-tuned alpha over training, verifying:
#  (a) it never exceeds the 0.1 cap,
#  (b) it doesn't collapse to ~0 (which would mean exploration died),
#  (c) and the final alpha value indicating how the SAC objective settled.
import os, glob
import matplotlib.pyplot as plt

# SB3 logs ent_coef in tensorboard under 'train/ent_coef'.  Load via
# tensorboard's event accumulator.
try:
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
    tb_dir = f'/kaggle/working/thesis/results/rl/sac_v216_seed{SEED}/tensorboard'
    runs = glob.glob(os.path.join(tb_dir, '*'))
    assert runs, f'No tensorboard runs in {tb_dir}'
    ea = EventAccumulator(runs[0])
    ea.Reload()
    tags = ea.Tags()['scalars']
    print('Available scalars:', tags)
    if 'train/ent_coef' in tags:
        events = ea.Scalars('train/ent_coef')
        steps = [e.step for e in events]
        vals  = [e.value for e in events]
        plt.figure(figsize=(10,4))
        plt.plot(steps, vals, '-', linewidth=1)
        plt.axhline(0.1, color='r', linestyle=':', label='cap = 0.1')
        plt.axhline(0.05, color='gray', linestyle='--', label='initial = 0.05')
        plt.axhline(0.01, color='blue', linestyle=':', label='v2.14/v2.15 fixed = 0.01')
        plt.xlabel('step'); plt.ylabel('ent_coef (alpha)')
        plt.title('v2.16 auto-tuned alpha trajectory (capped at 0.1)')
        plt.yscale('log'); plt.legend(); plt.grid(alpha=0.3)
        plt.tight_layout(); plt.show()
        print(f'Alpha range over training: [{min(vals):.4e}, {max(vals):.4e}]')
        print(f'Final alpha:               {vals[-1]:.4e}')
        print(f'Cap violations (>0.1):     {sum(1 for v in vals if v > 0.1)}')
except Exception as e:
    print('Tensorboard trajectory unavailable:', e)


In [ ]:
# Cell 10: Resume from a saved checkpoint (if the session was interrupted).
# Upload the prior output as a dataset, then fill in the path.

# SEED = 0
# CHECKPOINT_STEP = 100_000
# CHECKPOINT_PATH = f'/kaggle/input/<your-dataset>/sac_v216_seed{SEED}/checkpoints/sac_v216_seed{SEED}_{CHECKPOINT_STEP}_steps.zip'
#
# from src.rl.train_v216 import CappedAutoAlphaAsymmetricLRSAC
# from src.rl.networks import V216CTDESACPolicy
# model = CappedAutoAlphaAsymmetricLRSAC.load(
#     CHECKPOINT_PATH, custom_objects={'policy_class': V216CTDESACPolicy})
# # Continue: model.learn(total_timesteps=..., reset_num_timesteps=False)
